# 05. VectorStoreRetrieverMemory → Store 의 **시맨틱 검색** (`index=`)

| legacy | LangGraph |
|---|---|
| `FAISS(...)` + `VectorStoreRetrieverMemory(retriever=vs.as_retriever(k=1))` | `InMemoryStore(index={"embed": embeddings, "dims": 1536})` |
| `memory.save_context({"human": ..}, {"ai": ..})` | `store.put(namespace, key, {"text": ...})` (저장 시 자동 임베딩) |
| `memory.load_memory_variables({"prompt": 질문})` | `store.search(namespace, query=질문, limit=1)` |

> `FAISS`, `as_retriever()` 같은 **벡터스토어 자체는 변경되지 않았습니다.** deprecated 된 것은 이를 감싸던 `VectorStoreRetrieverMemory` 클래스입니다.

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [2]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")  # 1536 차원

interview = [
    ("안녕하세요, 오늘 면접에 참석해주셔서 감사합니다. 자기소개 부탁드립니다.",
     "안녕하세요. 저는 컴퓨터 과학을 전공한 신입 개발자입니다. 대학에서는 주로 자바와 파이썬을 사용했으며, 최근에는 웹 개발 프로젝트에 참여했습니다."),
    ("프로젝트에서 어떤 역할을 맡았나요?",
     "제가 맡은 역할은 백엔드 개발자였습니다. 사용자 데이터 처리와 서버 로직 개발을 담당했고, RESTful API 를 구현했습니다. 데이터베이스는 MySQL 을 사용했습니다."),
    ("팀 프로젝트에서 어려움을 겪었던 경험이 있다면 어떻게 해결했나요?",
     "초기에 의사소통 문제가 있었습니다. 정기적인 회의를 제안하고 각자의 진행 상황을 공유하도록 해서 해결했습니다."),
    ("개발자로서 자신의 강점은 무엇이라고 생각하나요?",
     "빠른 학습 능력과 문제 해결 능력입니다. 새로운 기술을 빠르게 습득해 적용할 수 있고, 복잡한 문제를 체계적으로 분석해 해결합니다."),
]
texts = [f"Human: {q}\nAI: {a}" for q, a in interview]

## 1. (변경 없음) 벡터스토어 + retriever 는 그대로 사용 가능

In [3]:
from langchain_community.vectorstores import FAISS

vs = FAISS.from_texts(texts, embeddings)
retriever = vs.as_retriever(search_kwargs={"k": 1})
print(retriever.invoke("면접자 전공은 무엇인가요?")[0].page_content)

C:\Users\user\AppData\Local\Temp\ipykernel_21448\1405058561.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Human: 안녕하세요, 오늘 면접에 참석해주셔서 감사합니다. 자기소개 부탁드립니다.
AI: 안녕하세요. 저는 컴퓨터 과학을 전공한 신입 개발자입니다. 대학에서는 주로 자바와 파이썬을 사용했으며, 최근에는 웹 개발 프로젝트에 참여했습니다.


## 2. LangGraph Store 로 대체

In [4]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore(index={"embed": embeddings, "dims": 1536, "fields": ["text"]})
ns = ("interview", "user-1")  # (용도, 사용자) 네임스페이스

# legacy: memory.save_context(inputs={"human": q}, outputs={"ai": a})
for i, text in enumerate(texts):
    store.put(ns, f"turn-{i}", {"text": text})

# legacy: memory.load_memory_variables({"prompt": "..."})["history"]
for query in ["면접자 전공은 무엇인가요?", "면접자가 프로젝트에서 맡은 역할은 무엇인가요?"]:
    hit = store.search(ns, query=query, limit=1)[0]
    print(f"Q: {query}\n  score={hit.score:.3f}\n  {hit.value['text']}\n")

Q: 면접자 전공은 무엇인가요?
  score=0.240
  Human: 안녕하세요, 오늘 면접에 참석해주셔서 감사합니다. 자기소개 부탁드립니다.
AI: 안녕하세요. 저는 컴퓨터 과학을 전공한 신입 개발자입니다. 대학에서는 주로 자바와 파이썬을 사용했으며, 최근에는 웹 개발 프로젝트에 참여했습니다.

Q: 면접자가 프로젝트에서 맡은 역할은 무엇인가요?
  score=0.473
  Human: 프로젝트에서 어떤 역할을 맡았나요?
AI: 제가 맡은 역할은 백엔드 개발자였습니다. 사용자 데이터 처리와 서버 로직 개발을 담당했고, RESTful API 를 구현했습니다. 데이터베이스는 MySQL 을 사용했습니다.



## 3. 그래프에 연결: 매 턴 "관련 기억 검색 → 답변 → 이번 턴 저장"

In [5]:
import uuid
from dataclasses import dataclass
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime


@dataclass
class Context:
    user_id: str


def chatbot(state: MessagesState, runtime: Runtime[Context]):
    ns = ("interview", runtime.context.user_id)
    question = state["messages"][-1].content

    hits = runtime.store.search(ns, query=question, limit=2)
    memories = "\n\n".join(h.value["text"] for h in hits)
    system = SystemMessage(f"다음은 관련된 과거 대화 기록입니다:\n{memories}\n\n이를 참고해 답하세요.")
    answer = llm.invoke([system, state["messages"][-1]])   # 원문 history 대신 검색된 기억만 사용

    runtime.store.put(ns, str(uuid.uuid4()), {"text": f"Human: {question}\nAI: {answer.content}"})
    return {"messages": [answer]}


graph = (
    StateGraph(MessagesState, context_schema=Context)
    .add_node("chatbot", chatbot)
    .add_edge(START, "chatbot")
    .compile(checkpointer=InMemorySaver(), store=store)
)

out = graph.invoke(
    {"messages": [HumanMessage("면접자가 사용한 데이터베이스는 뭐였나요?")]},
    config={"configurable": {"thread_id": "t-1"}},
    context={"user_id": "user-1"},
)
print(out["messages"][-1].content)
print("저장된 기억 수:", len(store.search(ns, limit=100)))

면접자가 사용한 데이터베이스는 MySQL입니다.
저장된 기억 수: 5


### 참고
* `InMemoryStore(index=...)` 는 실습용입니다. 운영에서는 `PostgresStore`(pgvector) 처럼 인덱스를 지원하는 영속 Store 를 사용합니다.
* 대화 원문(단기 기억)은 checkpointer, 검색용 기억(장기 기억)은 Store 로 **역할이 분리**됩니다.